In [1]:
import numpy as np
from PIL import Image
from typing import Dict, List, Optional
import struct

def read_jpeg_parameters(image_path: str) -> Dict:
    """
    Wczytuje plik JPEG i odczytuje jego tablice kwantyzacji oraz inne parametry.
    """
    try:
        img = Image.open(image_path)
        
        parameters = {
            'filepath': image_path,
            'image_size': (img.height, img.width),
            'color_space': img.mode,
            'quant_tables': {},
            'subsampling': None,
            'progressive': False
        }
        
        # Odczytanie tablic kwantyzacji z pliku
        with open(image_path, 'rb') as f:
            data = f.read()
            
        # Szukanie markerów DQT (Define Quantization Table) - 0xFFDB
        i = 0
        table_idx = 0
        while i < len(data) - 1:
            if data[i] == 0xFF and data[i+1] == 0xDB:  # DQT marker
                i += 2
                length = struct.unpack('>H', data[i:i+2])[0]
                i += 2
                
                # Odczyt tablicy kwantyzacji
                precision_and_id = data[i]
                precision = (precision_and_id >> 4) & 0x0F  # 0 = 8-bit, 1 = 16-bit
                table_id = precision_and_id & 0x0F
                i += 1
                
                if precision == 0:  # 8-bit values
                    table_size = 64
                    table = struct.unpack('B' * table_size, data[i:i+table_size])
                else:  # 16-bit values
                    table_size = 128
                    table = struct.unpack('>' + 'H' * 64, data[i:i+table_size])
                
                # Przekształcenie do tablicy 8x8
                qt_array = np.array(table).reshape(8, 8)
                parameters['quant_tables'][f'table_{table_id}'] = qt_array
                
                i += table_size
                table_idx += 1
            else:
                i += 1
        
        # Sprawdzenie czy obraz jest progressive
        i = 0
        while i < len(data) - 1:
            if data[i] == 0xFF and data[i+1] == 0xC2:  # SOF2 (Progressive DCT)
                parameters['progressive'] = True
                break
            i += 1
        
        # Odczytanie informacji o próbkowaniu z Pillow
        if hasattr(img, 'layer'):
            parameters['subsampling'] = img.layer
        
        return parameters
        
    except Exception as e:
        print(f"Błąd podczas wczytywania pliku {image_path}: {str(e)}")
        return None


def print_jpeg_info(parameters: Dict):
    """
    Wyświetla informacje o parametrach JPEG w czytelny sposób.
    """
    if parameters is None:
        print("Brak danych do wyświetlenia")
        return
    
    print(f"\n{'='*60}")
    print(f"Informacje o pliku JPEG: {parameters['filepath']}")
    print(f"{'='*60}")
    print(f"Rozmiar obrazu: {parameters['image_size'][0]} x {parameters['image_size'][1]} px")
    print(f"Przestrzeń kolorów: {parameters['color_space']}")
    print(f"Progressive JPEG: {'Tak' if parameters['progressive'] else 'Nie'}")
    
    if parameters['subsampling']:
        print(f"Próbkowanie: {parameters['subsampling']}")
    
    print(f"\nTablice kwantyzacji (znaleziono: {len(parameters['quant_tables'])}):")
    for name, table in parameters['quant_tables'].items():
        print(f"\n{name}:")
        print(table)


# Przykład użycia
if __name__ == "__main__":
    # Podaj ścieżkę do pliku JPEG
    image_path = "test-images/2-min.jpg"
    
    params = read_jpeg_parameters(image_path)
    if params:
        print_jpeg_info(params)


Informacje o pliku JPEG: test-images/2-min.jpg
Rozmiar obrazu: 1024 x 1024 px
Przestrzeń kolorów: RGB
Progressive JPEG: Tak
Próbkowanie: [(1, 2, 2, 0), (2, 1, 1, 1), (3, 1, 1, 1)]

Tablice kwantyzacji (znaleziono: 1):

table_0:
[[ 4  4  4  4  4  4  4  4]
 [ 4  4  6  6  5  6  6  8]
 [ 7  7  7  7  8 12  9  9]
 [ 9  9  9 12 19 12 14 12]
 [12 14 12 19 17 20 16 15]
 [16 20 17 30 23 21 21 23]
 [30 34 29 27 29 34 42 37]
 [37 42 52 50 52 68 68 92]]


# Test obrazów skompresowanych kilkukrotnie z tą samą jakością

In [4]:
print_jpeg_info(read_jpeg_parameters("multiple-compression-same-quality/01 (1)_q70.jpg"))
print_jpeg_info(read_jpeg_parameters("multiple-compression-same-quality/01 (1)_q70.jpg.q70.jpg"))
print_jpeg_info(read_jpeg_parameters("multiple-compression-same-quality/01 (1)_q70.jpg.q70.q70.jpg"))


Informacje o pliku JPEG: multiple-compression-same-quality/01 (1)_q70.jpg
Rozmiar obrazu: 768 x 512 px
Przestrzeń kolorów: RGB
Progressive JPEG: Nie
Próbkowanie: [(1, 2, 2, 0), (2, 1, 1, 1), (3, 1, 1, 1)]

Tablice kwantyzacji (znaleziono: 2):

table_0:
[[10  7  7  8  7  6 10  8]
 [ 8  8 11 10 10 11 14 24]
 [16 14 13 13 14 29 21 22]
 [17 24 35 31 37 36 34 31]
 [34 33 38 43 55 47 38 41]
 [52 41 33 34 48 65 49 52]
 [57 59 62 62 62 37 46 68]
 [73 67 60 72 55 61 62 59]]

table_1:
[[10 11 11 14 13 14 28 16]
 [16 28 59 40 34 40 59 59]
 [59 59 59 59 59 59 59 59]
 [59 59 59 59 59 59 59 59]
 [59 59 59 59 59 59 59 59]
 [59 59 59 59 59 59 59 59]
 [59 59 59 59 59 59 59 59]
 [59 59 59 59 59 59 59 59]]

Informacje o pliku JPEG: multiple-compression-same-quality/01 (1)_q70.jpg.q70.jpg
Rozmiar obrazu: 768 x 512 px
Przestrzeń kolorów: RGB
Progressive JPEG: Nie
Próbkowanie: [(1, 2, 2, 0), (2, 1, 1, 1), (3, 1, 1, 1)]

Tablice kwantyzacji (znaleziono: 2):

table_0:
[[10  7  7  8  7  6 10  8]
 [ 8  8 11 10

In [6]:
# porownanie tablic kwantyzacji dla kilku wynikow
params1 = read_jpeg_parameters("multiple-compression-same-quality/01 (1)_q70.jpg")
params2 = read_jpeg_parameters("multiple-compression-same-quality/01 (1)_q70.jpg.q70.jpg")
params3 = read_jpeg_parameters("multiple-compression-same-quality/01 (1)_q70.jpg.q70.q70.jpg")

print("\n" + "="*60)
print("Porównanie tablic kwantyzacji")
print("="*60)

# Porównanie dla każdej tablicy kwantyzacji
for table_name in params1['quant_tables'].keys():
    print(f"\n{table_name}:")
    print("\n1x kompresja (q70):")
    print(params1['quant_tables'][table_name])
    print("\n2x kompresja (q70):")
    print(params2['quant_tables'][table_name])
    print("\n3x kompresja (q70):")
    print(params3['quant_tables'][table_name])
    
    # Różnice między kolejnymi kompresami
    diff_1_2 = params2['quant_tables'][table_name] - params1['quant_tables'][table_name]
    diff_2_3 = params3['quant_tables'][table_name] - params2['quant_tables'][table_name]
    
    print(f"\nRóżnica (2x - 1x):")
    print(diff_1_2)
    print(f"\nRóżnica (3x - 2x):")
    print(diff_2_3)


Porównanie tablic kwantyzacji

table_0:

1x kompresja (q70):
[[10  7  7  8  7  6 10  8]
 [ 8  8 11 10 10 11 14 24]
 [16 14 13 13 14 29 21 22]
 [17 24 35 31 37 36 34 31]
 [34 33 38 43 55 47 38 41]
 [52 41 33 34 48 65 49 52]
 [57 59 62 62 62 37 46 68]
 [73 67 60 72 55 61 62 59]]

2x kompresja (q70):
[[10  7  7  8  7  6 10  8]
 [ 8  8 11 10 10 11 14 24]
 [16 14 13 13 14 29 21 22]
 [17 24 35 31 37 36 34 31]
 [34 33 38 43 55 47 38 41]
 [52 41 33 34 48 65 49 52]
 [57 59 62 62 62 37 46 68]
 [73 67 60 72 55 61 62 59]]

3x kompresja (q70):
[[10  7  7  8  7  6 10  8]
 [ 8  8 11 10 10 11 14 24]
 [16 14 13 13 14 29 21 22]
 [17 24 35 31 37 36 34 31]
 [34 33 38 43 55 47 38 41]
 [52 41 33 34 48 65 49 52]
 [57 59 62 62 62 37 46 68]
 [73 67 60 72 55 61 62 59]]

Różnica (2x - 1x):
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

Różnica (3x - 2x):
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 

# Podsumowując, nie ma różnic jeżeli obraz jest kompresowany do tej samej jakości wielokrotnie

In [7]:
def read_huffman_tables(image_path: str) -> Dict:
    """
    Odczytuje tablice Huffmana z pliku JPEG.
    """
    huffman_tables = {
        'DC': {},  # DC tables
        'AC': {}   # AC tables
    }
    
    try:
        with open(image_path, 'rb') as f:
            data = f.read()
        
        # Szukanie markerów DHT (Define Huffman Table) - 0xFFC4
        i = 0
        while i < len(data) - 1:
            if data[i] == 0xFF and data[i+1] == 0xC4:  # DHT marker
                i += 2
                length = struct.unpack('>H', data[i:i+2])[0]
                i += 2
                
                end_pos = i + length - 2
                
                while i < end_pos:
                    # Odczyt informacji o tablicy
                    table_info = data[i]
                    table_class = (table_info >> 4) & 0x0F  # 0 = DC, 1 = AC
                    table_id = table_info & 0x0F
                    i += 1
                    
                    # Odczyt liczby kodów o różnych długościach (16 bajtów)
                    num_codes = list(data[i:i+16])
                    i += 16
                    
                    # Odczyt wartości symboli
                    total_codes = sum(num_codes)
                    symbols = list(data[i:i+total_codes])
                    i += total_codes
                    
                    # Zapisanie tablicy
                    table_type = 'DC' if table_class == 0 else 'AC'
                    huffman_tables[table_type][f'table_{table_id}'] = {
                        'num_codes': num_codes,
                        'symbols': symbols,
                        'total_codes': total_codes
                    }
            else:
                i += 1
        
        return huffman_tables
        
    except Exception as e:
        print(f"Błąd podczas odczytywania tablic Huffmana: {str(e)}")
        return None


def print_huffman_info(huffman_tables: Dict, image_path: str = ""):
    """
    Wyświetla informacje o tablicach Huffmana.
    """
    if huffman_tables is None:
        print("Brak danych do wyświetlenia")
        return
    
    print(f"\n{'='*60}")
    print(f"Tablice Huffmana: {image_path}")
    print(f"{'='*60}")
    
    for table_type in ['DC', 'AC']:
        print(f"\n{table_type} Tables:")
        for table_name, table_data in huffman_tables[table_type].items():
            print(f"\n  {table_name}:")
            print(f"    Liczba kodów według długości: {table_data['num_codes']}")
            print(f"    Łączna liczba kodów: {table_data['total_codes']}")
            print(f"    Symbole (pierwsze 20): {table_data['symbols'][:20]}")


# Test odczytu tablic Huffmana
huffman1 = read_huffman_tables("multiple-compression-same-quality/01 (1)_q70.jpg")
print_huffman_info(huffman1, "01 (1)_q70.jpg")

huffman2 = read_huffman_tables("multiple-compression-same-quality/01 (1)_q70.jpg.q70.jpg")
print_huffman_info(huffman2, "01 (1)_q70.jpg.q70.jpg")


Tablice Huffmana: 01 (1)_q70.jpg

DC Tables:

  table_0:
    Liczba kodów według długości: [0, 1, 5, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
    Łączna liczba kodów: 12
    Symbole (pierwsze 20): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

  table_1:
    Liczba kodów według długości: [0, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
    Łączna liczba kodów: 12
    Symbole (pierwsze 20): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

AC Tables:

  table_0:
    Liczba kodów według długości: [0, 2, 1, 3, 3, 2, 4, 3, 5, 5, 4, 4, 0, 0, 1, 125]
    Łączna liczba kodów: 162
    Symbole (pierwsze 20): [1, 2, 3, 0, 4, 17, 5, 18, 33, 49, 65, 6, 19, 81, 97, 7, 34, 113, 20, 50]

  table_1:
    Liczba kodów według długości: [0, 2, 1, 2, 4, 4, 3, 4, 7, 5, 4, 4, 0, 1, 2, 119]
    Łączna liczba kodów: 162
    Symbole (pierwsze 20): [0, 1, 2, 3, 17, 4, 5, 33, 49, 6, 18, 65, 81, 7, 97, 113, 19, 34, 50, 129]

Tablice Huffmana: 01 (1)_q70.jpg.q70.jpg

DC Tables:

  table_0:
    Liczba kodów według długości: [0, 1, 5,

W kontekście tablic Huffmana w JPEG, AC i DC oznaczają te same współczynniki co wcześniej, ale są kodowane różnymi tablicami Huffmana ze względu na ich odmienne właściwości statystyczne:

DC Huffman Tables
Kodują współczynniki DC (średnie wartości bloków 8×8)
Zazwyczaj kodują różnicę między współczynnikiem DC bieżącego bloku a poprzedniego (DPCM - Differential Pulse Code Modulation)
Różnice DC są zwykle małe, ponieważ sąsiednie bloki mają podobną jasność
Mają mniejsze tablice - typowo 12-16 symboli

AC Huffman Tables
Kodują współczynniki AC (63 współczynniki częstotliwościowe w każdym bloku)
Kodują pary (run-length, value): długość serii zer + wartość współczynnika
Wiele współczynników AC to zera (szczególnie wysokich częstotliwości)
Mają większe tablice - typowo 162-256 symboli
W kodzie:
```
huffman_tables = {
    'DC': {},  # Tablice do kodowania współczynników DC
    'AC': {}   # Tablice do kodowania współczynników AC
}
```
Typowy plik JPEG zawiera 4 tablice Huffmana:

DC table_0 - dla składowej luminancji (Y)
DC table_1 - dla składowych chrominancji (Cb, Cr)
AC table_0 - dla składowej luminancji (Y)
AC table_1 - dla składowych chrominancji (Cb, Cr)